# Zadaca 1 - Imad Buljic

Ovaj notebook rjesava dva trazena dijela zadace nad Iris skupom podataka.
Prvi dio zavrsava Random Search za odabir hiperparametara SVM modela,
a drugi dio pretvara pseudokod algoritma sismisa u Python kod koji radi
pretragu prostora hiperparametara.


In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.exceptions import ConvergenceWarning

np.random.seed(42)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
plt.style.use("seaborn-v0_8-whitegrid")


: 

## Ucitavanje podataka

Koristi se lokalni fajl `iris.csv` iz foldera zadace. Nakon ucitavanja se odvajaju
ulazne varijable i ciljna kolona, a za evaluaciju se priprema stratifikovani 5-fold CV.


In [ ]:
df = pd.read_csv("iris.csv")

X = df.drop("species", axis=1)
y = df["species"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Broj redova i kolona: {df.shape}")
print(f"Nazivi klasa: {sorted(y.unique().tolist())}")
print("Broj uzoraka po klasi:")
print(y.value_counts())
print()

display(df.head())


## Random Search

U vjezbama je Random Search bio zapocet, ali je medju parametrima stajao i `tt_split`.
Taj parametar ne pripada klasi `SVC`, pa ovdje ostavljamo samo one opcije koje model
zaista podrzava. Pretraga se izvodi nad pipeline-om koji prvo standardizuje podatke,
pa zatim trenira SVM.


In [ ]:
parametri_random = {
    "svc__C": [0.1, 0.5, 1, 2, 5, 10, 20, 50],
    "svc__kernel": ["linear", "rbf"],
    "svc__gamma": ["scale", "auto", 0.001, 0.01, 0.1, 1],
    "svc__class_weight": [None, "balanced"],
    "svc__shrinking": [True, False],
    "svc__tol": [1e-3, 1e-4],
    "svc__decision_function_shape": ["ovr"],
    "svc__max_iter": [-1, 500, 1000]
}

pipeline_random = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC())
])

random_search = RandomizedSearchCV(
    estimator=pipeline_random,
    param_distributions=parametri_random,
    n_iter=50,
    scoring="f1_weighted",
    cv=cv,
    random_state=42,
    n_jobs=1
)

start_random = time.time()
random_search.fit(X, y)
random_vrijeme = time.time() - start_random

print("Najbolji parametri za Random Search:")
for naziv, vrijednost in random_search.best_params_.items():
    print(f"  {naziv}: {vrijednost}")
print(f"Najbolji F1 score: {random_search.best_score_:.4f}")
print(f"Vrijeme izvrsavanja: {random_vrijeme:.2f} s")


## Algoritam sismisa - pseudokod i ideja

Pseudokod sa vjezbi se moze sazeti ovako:

```
za svaki sismis i:
    beta = random(0, 1)
    frekvencija = f_min + (f_max - f_min) * beta
    brzina[i] = brzina[i] + (pozicija[i] - best) * frekvencija
    nova_pozicija = pozicija[i] + brzina[i]

    ako random(0, 1) > pulse_rate[i]:
        nova_pozicija = best + epsilon * prosjecna_glasnoca

    nova_pozicija = popravi_granice(nova_pozicija)
    novi_fitness = objective(nova_pozicija)

    ako novi_fitness < fitness[i] i random(0, 1) < loudness[i]:
        prihvati novu poziciju
        smanji glasnocu
```

U ovoj implementaciji jedan sismis predstavlja jednu kombinaciju SVM hiperparametara.
Pozicija je kontinualni vektor, a posebnim dekodiranjem se pretvara u konkretne vrijednosti
za `C`, `gamma` i `kernel`.


In [ ]:
DONJE_GRANICE = np.array([-2.0, -3.0, 0.0])
GORNJE_GRANICE = np.array([2.0, 1.0, 1.0])
KERNELI = ["linear", "rbf"]

def dekodiraj_parametre(pozicija):
    log_c = float(np.clip(pozicija[0], DONJE_GRANICE[0], GORNJE_GRANICE[0]))
    log_gamma = float(np.clip(pozicija[1], DONJE_GRANICE[1], GORNJE_GRANICE[1]))
    indeks_kernela = int(np.clip(np.round(pozicija[2]), 0, len(KERNELI) - 1))
    kernel = KERNELI[indeks_kernela]

    return {
        "C": 10 ** log_c,
        "gamma": 10 ** log_gamma if kernel == "rbf" else "scale",
        "kernel": kernel,
    }


def sredi_granice(pozicija):
    return np.clip(pozicija, DONJE_GRANICE, GORNJE_GRANICE)


def ciljna_funkcija(pozicija, X, y, cv, cache):
    parametri = dekodiraj_parametre(pozicija)
    gamma_kljuc = parametri["gamma"] if isinstance(parametri["gamma"], str) else round(parametri["gamma"], 8)
    kljuc = (round(parametri["C"], 8), gamma_kljuc, parametri["kernel"])

    if kljuc not in cache:
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(
                C=parametri["C"],
                gamma=parametri["gamma"],
                kernel=parametri["kernel"],
                decision_function_shape="ovr"
            ))
        ])
        score = cross_val_score(model, X, y, cv=cv, scoring="f1_weighted", n_jobs=1).mean()
        cache[kljuc] = -score

    return cache[kljuc]


## Glavna funkcija algoritma sismisa

Pretraga se vodi kroz populaciju sismisa. Svaki clan populacije ima svoju poziciju,
brzinu, glasnocu i stopu pulsa. Najbolje do tada pronadjeno rjesenje usmjerava dalji tok
pretrage.


In [ ]:
def bat_pretraga(X, y, cv, broj_sismisa=10, broj_iteracija=20, f_min=0.0, f_max=2.0, alpha=0.9):
    broj_dimenzija = len(DONJE_GRANICE)
    cache = {}

    pozicije = np.random.uniform(DONJE_GRANICE, GORNJE_GRANICE, size=(broj_sismisa, broj_dimenzija))
    brzine = np.zeros((broj_sismisa, broj_dimenzija))
    glasnoce = np.full(broj_sismisa, 0.9)
    pulse_rate = np.full(broj_sismisa, 0.2)

    fitness = np.array([ciljna_funkcija(pozicije[i], X, y, cv, cache) for i in range(broj_sismisa)])
    broj_evaluacija = broj_sismisa

    najbolji_indeks = np.argmin(fitness)
    najbolja_pozicija = pozicije[najbolji_indeks].copy()
    najbolji_fitness = fitness[najbolji_indeks]
    history = [-najbolji_fitness]

    for iteracija in range(broj_iteracija):
        prosjecna_glasnoca = np.mean(glasnoce)

        for i in range(broj_sismisa):
            beta = np.random.random()
            frekvencija = f_min + (f_max - f_min) * beta

            brzine[i] = brzine[i] + (pozicije[i] - najbolja_pozicija) * frekvencija
            nova_pozicija = pozicije[i] + brzine[i]

            if np.random.random() > pulse_rate[i]:
                epsilon = np.random.uniform(-1, 1, broj_dimenzija)
                nova_pozicija = najbolja_pozicija + epsilon * prosjecna_glasnoca

            nova_pozicija = sredi_granice(nova_pozicija)
            novi_fitness = ciljna_funkcija(nova_pozicija, X, y, cv, cache)
            broj_evaluacija += 1

            if novi_fitness < fitness[i] and np.random.random() < glasnoce[i]:
                pozicije[i] = nova_pozicija
                fitness[i] = novi_fitness
                glasnoce[i] = alpha * glasnoce[i]

            if fitness[i] < najbolji_fitness:
                najbolja_pozicija = pozicije[i].copy()
                najbolji_fitness = fitness[i]

        history.append(-najbolji_fitness)

    najbolji_parametri = dekodiraj_parametre(najbolja_pozicija)
    return najbolji_parametri, -najbolji_fitness, history, broj_evaluacija


## Pokretanje algoritma sismisa

Za izvrsavanje se koristi umjeren broj sismisa i iteracija kako bi pretraga ostala jasna,
a notebook se i dalje mogao brzo pokrenuti od pocetka do kraja.


In [ ]:
np.random.seed(42)

start_bat = time.time()
bat_parametri, bat_score, bat_history, bat_evaluacije = bat_pretraga(
    X, y, cv, broj_sismisa=8, broj_iteracija=15, f_min=0.0, f_max=2.0, alpha=0.9
)
bat_vrijeme = time.time() - start_bat

print("Najbolji parametri za algoritam sismisa:")
for naziv, vrijednost in bat_parametri.items():
    print(f"  {naziv}: {vrijednost}")
print(f"Najbolji F1 score: {bat_score:.4f}")
print(f"Vrijeme izvrsavanja: {bat_vrijeme:.2f} s")
print(f"Broj evaluacija: {bat_evaluacije}")


## Poredjenje metoda

Na kraju se uporedjuju kvalitet rjesenja, vrijeme izvrsavanja i broj evaluacija.
Pored tabele se crta i graf koji pokazuje kako je algoritam sismisa napredovao kroz iteracije.


In [ ]:
rezultati = pd.DataFrame({
    "Metoda": ["Random Search", "Algoritam sismisa"],
    "Najbolji F1": [round(random_search.best_score_, 4), round(bat_score, 4)],
    "Vrijeme (s)": [round(random_vrijeme, 2), round(bat_vrijeme, 2)],
    "Broj evaluacija": [random_search.n_iter, bat_evaluacije]
})

display(rezultati)

plt.figure(figsize=(10, 5))
plt.plot(range(len(bat_history)), bat_history, marker="o", linewidth=2, label="Algoritam sismisa")
plt.axhline(random_search.best_score_, color="crimson", linestyle="--", label="Random Search")
plt.xlabel("Iteracija")
plt.ylabel("Najbolji F1 score")
plt.title("Konvergencija algoritma sismisa i rezultat Random Search metode")
plt.legend()
plt.tight_layout()
plt.show()


## Zakljucak

Random Search je jednostavan nacin da se nasumicno isprobaju razlicite kombinacije
hiperparametara i brzo dobije dobar rezultat. Algoritam sismisa ide korak dalje jer kroz
iteracije koristi trenutno najbolje rjesenje da usmjeri novu pretragu. Na Iris skupu podataka
obe metode mogu dati vrlo jak rezultat, ali algoritam sismisa dodatno pokazuje ideju
metaheuristicke optimizacije kroz pracenje konvergencije.
